# Готовые результаты: лучи и обход стены

Этот ноутбук читает опубликованные таблицы и сохранённые кадры. Обучение, загрузка весов и новая генерация здесь не запускаются.

В виджетах показаны **фиксированные иллюстрации**: условие 0, первые 16 шумовых входов, новый training seed 161 и общий родитель. У лучей это target 0, у стены — scene 0. Полные количественные таблицы учитывают **все условия и три training seed**; иллюстрации не служили выбором лучшей модели.

Сравнивайте методы при одинаковом NFE — числе вызовов сети на один объект. Кадры consistency-модели включают настоящие события denoise → re-noise; их время немонотонно. У стены физическое движение вдоль готового маршрута и шаги генерации целого маршрута — разные анимации.

Основное сравнение использует **3 training seed: 161, 262 и 363**. Общий бюджет продолжения обучения — 2000 обновлений на метод. У progressive показаны заранее заданные снимки: NFE 4 после **1200**, NFE 2 после **1600**, NFE 1 после **2000** суммарных обновлений; учитывайте эту накопленную стоимость при сравнении. Исторические модели обучались 1000 обновлений и вынесены в отдельную группу.

Модель **endpoint** училась выдавать конечную точку за один вызов. Её строки и иллюстрации при NFE 2/4 отмечены **off-target**; это диагностическое применение вне целевого режима, а не отдельная модель, обученная на 2/4 шага. Числа в интерфейсе округлены до пяти значащих цифр; исходные CSV/JSON сохраняют полную точность.


In [ ]:
import os
import subprocess
import sys
from importlib.metadata import version

from packaging.version import Version

GIT_BRANCH = "research/wasserstein-tube-replication"
GIT_URL = "https://github.com/Vovanm88/ddpm_vibe_research.git"
if os.environ.get("DIFFUSION_LAB_SKIP_INSTALL") != "1":
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--upgrade",
            "diffusion-lab[replication,interactive] @ git+" + GIT_URL + "@" + GIT_BRANCH,
        ]
    )
if Version(version("diffusion-lab")) < Version("0.5.0"):
    raise RuntimeError("Install diffusion-lab >=0.5.0 from this branch, then restart the kernel")


In [ ]:
import importlib.util
import tempfile
from pathlib import Path

# Reuse a local checkout, including a diffusion-lab/ child of the current project.
REPO_ROOT = None
for parent in (Path.cwd(), *Path.cwd().parents):
    for candidate in (parent, parent / "diffusion-lab"):
        if (candidate / "examples/evaluate_method_zoo.py").is_file():
            REPO_ROOT = candidate.resolve()
            break
    if REPO_ROOT is not None:
        break
if REPO_ROOT is None:
    REPO_ROOT = Path(tempfile.mkdtemp(prefix="method-zoo-results-")) / "repository"
    subprocess.check_call(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            GIT_BRANCH,
            GIT_URL,
            str(REPO_ROOT),
        ]
    )
print("Репозиторий:", REPO_ROOT)


## Интерактивный просмотр

Выберите задачу, затем Method, NFE, Noise и Event. Play проигрывает только сохранённые кадры, по одному событию за шаг. Возврат к задаче сохраняет её настройки.

Для лучей меньше conditional SWD означает лучшее совпадение распределения внутри каждого запрошенного класса; routing и покрытие внешней части лучей нужно читать вместе. Для стены сначала смотрите успешность без столкновений и наличие обоих маршрутов. Столбец Pareto не сводит разные свойства к единому баллу. Интервалы по трём seed — разведочная оценка неопределённости, а не гарантия превосходства.

In [ ]:
import csv
import html
import json
import math

import ipywidgets as widgets
from IPython.display import HTML, Image, clear_output, display


def load_evaluation_core(repo_root):
    path = Path(repo_root) / "examples/evaluate_method_zoo.py"
    spec = importlib.util.spec_from_file_location("cached_method_zoo_results", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def _csv_rows(path):
    if not path.is_file():
        return []
    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))


def _method_label(method, nfe):
    try:
        off_target = method == "endpoint" and int(nfe) in (2, 4)
    except (TypeError, ValueError):
        off_target = False
    return "endpoint (off-target)" if off_target else str(method)


def _format_cell(value):
    text = str(value)
    try:
        number = float(text)
    except (TypeError, ValueError):
        return html.escape(text)
    if not math.isfinite(number) or text.lstrip("+-").isdigit():
        return html.escape(text)
    return html.escape(format(number, ".5g"))


def _table(rows, columns):
    if not rows:
        return HTML("<p>Таблица пока не опубликована.</p>")
    columns = [key for key in columns if any(key in row for row in rows)]
    header = "".join("<th>" + html.escape(key) + "</th>" for key in columns)
    body = "".join(
        "<tr>"
        + "".join(
            "<td>"
            + _format_cell(
                _method_label(row.get(key, ""), row.get("nfe"))
                if key == "method"
                else row.get(key, "")
            )
            + "</td>"
            for key in columns
        )
        + "</tr>"
        for row in rows
    )
    return HTML(
        '<div style="overflow-x:auto"><table><thead><tr>'
        + header
        + "</tr></thead><tbody>"
        + body
        + "</tbody></table></div>"
    )


def _annotate_preview(view):
    # The cached renderer stays unchanged; this caption clarifies the selected mode.
    method, nfe = view.children[0].children[:2]
    marker = widgets.HTML()

    def show_mode(change=None):
        marker.value = (
            "<b>endpoint (off-target)</b>: модель училась предсказывать конечную точку за один вызов; NFE 2/4 — применение вне целевого режима."
            if _method_label(method.value, nfe.value).endswith("(off-target)")
            else ""
        )

    method.observe(show_mode, names="value")
    nfe.observe(show_mode, names="value")
    show_mode()
    return widgets.VBox([marker, view])


def create_results_browser(repo_root, *, core_module=None):
    repo_root = Path(repo_root)
    report = repo_root / "replication/reports/method_zoo/final"
    result_path = report / "results.json"
    results = json.loads(result_path.read_text()) if result_path.is_file() else None
    completed = results is not None and results.get("metadata", {}).get("status") == "completed"
    core = load_evaluation_core(repo_root) if core_module is None else core_module
    summary = _csv_rows(report / "primary_summary.csv")
    references = _csv_rows(report / "historical_and_parent.csv")
    selector = widgets.Dropdown(
        options=[("Вращающиеся лучи", "rays"), ("Обход стены", "wall")],
        value="rays",
        description="Задача",
    )
    tables, preview = widgets.Output(), widgets.Output()
    cached_views = {}
    columns = (
        "method",
        "nfe",
        "primary_metric",
        "primary_point_rank",
        "pareto_layer",
        "conditional_swd",
        "routing_accuracy_outer",
        "outer_mass_gap",
        "success_fraction",
        "both_modes_all_conditions",
    )

    def show_task(change=None):
        task = selector.value
        with tables:
            clear_output(wait=True)
            if not completed:
                display(
                    HTML(
                        "<p>Итоговый отчёт ещё не опубликован целиком. Обновите Git checkout и повторно выполните эту ячейку после публикации. Новые модели и данные не создаются.</p>"
                    )
                )
            else:
                display(HTML("<h4>Основное сравнение: все условия, три training seed</h4>"))
                display(_table([row for row in summary if row.get("task") == task], columns))
                reference_table = _table(
                    [row for row in references if row.get("task") == task],
                    (
                        "method",
                        "nfe",
                        "league",
                        "training_updates",
                        "conditional_swd",
                        "routing_accuracy_outer",
                        "success_fraction",
                        "both_modes_all_conditions",
                    ),
                )
                display(
                    HTML(
                        "<details><summary>Родитель и исторические модели — отдельная группа с другими бюджетами обучения</summary>"
                        + reference_table.data
                        + "</details>"
                    )
                )
        with preview:
            clear_output(wait=True)
            path = report / "previews" / f"{task}_preview.pt"
            if not completed or not path.is_file():
                display(HTML("<p>Сохранённые иллюстрации пока недоступны в этом checkout.</p>"))
                return
            if task not in cached_views:
                cached_views[task] = _annotate_preview(core.sampling_preview_widget(path))
            display(cached_views[task])

    selector.observe(show_task, names="value")
    show_task()
    return widgets.VBox(
        [
            widgets.HTML(
                "<b>Только сохранённые иллюстрации:</b> condition 0 · первые 16 noise · seed 161 + parent"
            ),
            selector,
            tables,
            preview,
        ]
    )


In [ ]:
BROWSER = create_results_browser(REPO_ROOT)
display(BROWSER)


## Сохранённые графики и GIF

Необязательная ячейка ниже показывает уже готовые файлы из того же Git checkout. Для полных значений доступны `primary_summary.csv`, `paired_vs_warm_cfm.csv`, `historical_and_parent.csv`, `per_seed.csv`, `per_condition.csv` и `ranking.json` в папке `replication/reports/method_zoo/final`.

Полные сгенерированные endpoints и истории не входят в этот ноутбук; небольшие публичные preview-файлы содержат только выбранные заранее иллюстрации. Происхождение моделей и файлов зафиксировано в `results.json`.

In [ ]:
SHOW_STATIC = False
if SHOW_STATIC:
    report = REPO_ROOT / "replication/reports/method_zoo/final"
    for task in ("rays", "wall"):
        for nfe in (1, 2, 4):
            image = report / "previews" / task / f"{task}_nfe{nfe}_final.png"
            animation = report / "previews" / task / f"{task}_nfe{nfe}_sampling_events.gif"
            for path in (image, animation):
                if path.is_file():
                    display(Image(filename=str(path)))
